# ABV 6 - Data Analysis II
## Pandas Advanced · Datenklasse · SQLite · KPI

Dieses Arbeitsblatt baut auf den bisherigen Datenanalyse-Aufgaben auf.
Sie arbeiten mit dem Datensatz `buchungen_2025.csv` und ueberfuehren ihn schrittweise in Auswertungen, ein kleineres Datenformat und eine SQLite-Datenbank.

| # | Thema | Methoden / Konzepte |
|---|-------|----------------------|
| 1 | Pandas Advanced I | `groupby`, `agg`, `apply`, neue Spalten |
| 2 | Pandas Advanced II | `pivot_table`, `merge`, neue Analysefelder |
| 3 | Datenklasse & Validierung | `dataclass`, Fehlerfall, optional `pydantic` |
| 4 | SQLite mit kleinem Datenformat | Tabellenstruktur, Insert pro Datensatz |
| HA | KPI aus Datenbank | `to_sql`, SQL-Read, Kennzahl, Vergleich mit neuem Datensatz |

Bearbeiten Sie die Aufgaben nacheinander. Unter jeder Aufgabe finden Sie aufklappbare Tipps.

In [2]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("buchungen_2025.csv")
df = pd.read_csv(DATA_PATH)
display(df.head(3))
print(df.shape)

,booking_id,booking_date,due_date,payment_date,fiscal_year,month,quarter,document_number,document_type,booking_type,...,amount_net,vat_rate,vat_amount,amount_gross,currency,is_paid,is_recurring,has_attachment,is_credit_note,notes
0,1,2025-05-21,2025-05-28,2025-06-24,2025,5,Q2,DOC-2025-0001,invoice,expense,...,629.38,0.19,119.58,748.96,EUR,True,False,True,False,NaN
1,2,2025-12-26,2026-01-25,2026-01-12,2025,12,Q4,DOC-2025-0002,invoice,revenue,...,1312.64,0.19,249.40,1562.04,EUR,True,False,True,False,Workshop
2,3,2025-07-14,2025-08-13,NaN,2025,7,Q3,DOC-2025-0003,invoice,revenue,...,1157.85,0.19,219.99,1377.84,EUR,False,False,False,False,NaN


(300, 31)


---
## Aufgabe 1 - Pandas Advanced I

### Aufgabe 1 - Gruppieren, aggregieren und Steuerbetrag berechnen

Arbeiten Sie mit den Buchungen aus `buchungen_2025.csv`.

**a)** Waehlen Sie nur Buchungen mit `vat_rate == 0.19` aus.

**b)** Erstellen Sie mit `apply(...)` eine neue Spalte, die den Bruttobetrag aus `amount_net` berechnet.  Diese gibt es eventuell schon, macht es trotzdem nochmal, nennt sie nur anders!

**c)** Gruppieren Sie die Daten anschliessend nach `category` und `booking_type`.

**d)** Erstellen Sie mit `groupby(...).agg(...)` mindestens diese Kennzahlen:
- Anzahl der Buchungen
- Summe `amount_net`
- Durchschnitt `amount_net`
- Summe des berechneten Bruttobetrags

**e)** Sortieren Sie das Ergebnis nach der Nettosumme absteigend.

<details>
<summary><b>Tipp zu a) - Mit Teilmenge starten</b></summary>

Speichern Sie die gefilterten Daten zuerst in einem neuen DataFrame, bevor Sie weitere Schritte darauf anwenden.
Beispielidee:
```python
df_vat_19 = df[df["vat_rate"] == 0.19].copy()
```
</details>

<details>
<summary><b>Tipp zu b) - apply fuer Zeilenwerte</b></summary>

Wenn Sie nur mit 19 Prozent arbeiten, reicht eine Funktion wie `x * 1.19`.
Achten Sie darauf, dass `amount_net` numerisch ist.
Beispielidee:
```python
df_vat_19["amount_net"] = pd.to_numeric(df_vat_19["amount_net"])
df_vat_19["amount_gross_calc"] = df_vat_19["amount_net"].apply(lambda x: x * 1.19)
```
</details>

<details>
<summary><b>Tipp zu c) und d) - groupby mit mehreren Aggregationen</b></summary>

`agg(...)` kann fuer eine Spalte mehrere Auswertungen gleichzeitig berechnen.
Sie koennen auch benannte Aggregationen verwenden.
Beispielidee:
```python
summary = df_vat_19.groupby(["category", "booking_type"]).agg(...)
```
</details>

<details>
<summary><b>Tipp zu e) - Ergebnis lesbar machen</b></summary>

Nach `groupby(...)` ist oft `reset_index()` hilfreich, bevor Sie sortieren oder das Ergebnis anzeigen.
</details>

In [ ]:
# Startpunkt fuer Aufgabe 1

df_vat_19 = df.copy()
display(df_vat_19.head(3))
# a) Filtern auf vat_rate == 0.19
df_vat_19 = df_vat_19[df_vat_19["vat_rate"] == 0.19]
# b) Berechnung der Bruttobeträge
brutto = df_vat_19['amount_net']
df_vat_19['brutto'] = brutto.apply(lambda x: x * 1.19)
#c)d) Gruppieren und Aggregieren
summary = df_vat_19.groupby(["category", "booking_type"]).agg(
    anzahl_buchungen=("amount_net", "count"),
    summe_netto=("amount_net", "sum"),
    durchschnitt_netto=("amount_net", "mean"),
    summe_brutto=("brutto", "sum")
)
#e) Reset des Index
summary = summary.reset_index()
display(summary.head(3))

,booking_id,booking_date,due_date,payment_date,fiscal_year,month,quarter,document_number,document_type,booking_type,...,amount_net,vat_rate,vat_amount,amount_gross,currency,is_paid,is_recurring,has_attachment,is_credit_note,notes
0,1,2025-05-21,2025-05-28,2025-06-24,2025,5,Q2,DOC-2025-0001,invoice,expense,...,629.38,0.19,119.58,748.96,EUR,True,False,True,False,NaN
1,2,2025-12-26,2026-01-25,2026-01-12,2025,12,Q4,DOC-2025-0002,invoice,revenue,...,1312.64,0.19,249.40,1562.04,EUR,True,False,True,False,Workshop
2,3,2025-07-14,2025-08-13,NaN,2025,7,Q3,DOC-2025-0003,invoice,revenue,...,1157.85,0.19,219.99,1377.84,EUR,False,False,False,False,NaN


,category,booking_type,anzahl_buchungen,summe_netto,durchschnitt_netto,summe_brutto
0,cogs,expense,28,30280.32,1081.440000,36033.5808
1,marketing,expense,33,13043.33,395.252424,15521.5627
2,other_income,revenue,50,61328.89,1226.577800,72981.3791


---
## Aufgabe 2 - Pandas Advanced II

### Aufgabe 2 - Pivot Table bauen und auf Realdaten mergen

In dieser Aufgabe erzeugen Sie zuerst eine Monatsauswertung und fuegen diese danach wieder an die Originaldaten an.

**a)** Erstellen Sie eine Pivot-Tabelle, die pro `month` die Summe von `amount_net` getrennt nach `booking_type` zeigt.

**b)** Benennen Sie die Spalten der Pivot-Tabelle so um, dass sie in den Originaldaten verstaendlich lesbar sind.

**c)** Mergen Sie die Pivot-Tabelle anschliessend wieder auf den Original-DataFrame.

**d)** Erstellen Sie auf dem gemergten DataFrame eine neue Spalte `monthly_result`, die Monatsumsatz und Monatsaufwand kombiniert.

**e)** Zeigen Sie anschliessend nur einige ausgewaehlte Spalten an, zum Beispiel `booking_id`, `month`, `booking_type`, `amount_net` und die neu gemergten Monatswerte.

<details>
<summary><b>Tipp zu a) - Pivot-Tabelle vorbereiten</b></summary>

Verwenden Sie `month` als Index und `booking_type` als Spalten.
Als Werte eignet sich `amount_net` mit einer Summenaggregation.
Beispielidee:
```python
pivot = df_months.pivot_table(
    index="month",
    columns="booking_type",
    values="amount_net",
    aggfunc="sum"
)
```
</details>

<details>
<summary><b>Tipp zu b) - Vor dem Merge reset_index()</b></summary>

Damit `month` wieder als normale Spalte verfuegbar ist, ist `reset_index()` meist notwendig.
Beispielidee:
```python
pivot = pivot.reset_index()
```
</details>

<details>
<summary><b>Tipp zu c) - Merge-Schluessel bewusst waehlen</b></summary>

Pruefen Sie genau, auf welcher Spalte beide DataFrames zusammenpassen.
In dieser Aufgabe ist das die Monatsnummer.
Beispielidee:
```python
merged = df_months.merge(pivot, on="month", how="left")
```
</details>

<details>
<summary><b>Tipp zu d) - Neue Kennzahl auf Basis des Merge</b></summary>

Wenn Sie die Umsatz- und Aufwandsspalten aus der Pivot-Tabelle erfolgreich gemergt haben, koennen Sie daraus eine Monatskennzahl berechnen.
</details>

In [30]:
# Startpunkt fuer Aufgabe 2
# a)
df_months = df[["booking_id", "month", "booking_type", "amount_net"]].copy()
df_months["amount_net"] = pd.to_numeric(df_months["amount_net"])
display(df_months.head(5))
tabelle = df_months.pivot_table(
    index="month",
    columns="booking_type",
    values="amount_net",
    aggfunc="sum",
)
# b)
tabelle = tabelle.reset_index()

# c)
zusammen = df_months.merge(tabelle, on="month", how="left")
display(zusammen)

,booking_id,month,booking_type,amount_net
0,1,5,expense,629.38
1,2,12,revenue,1312.64
2,3,7,revenue,1157.85
3,4,11,revenue,1976.63
4,5,3,expense,123.65


,booking_id,month,booking_type,amount_net,expense,revenue
0,1,5,expense,629.38,10509.73,20089.33
1,2,12,revenue,1312.64,11551.68,15514.99
2,3,7,revenue,1157.85,4745.44,21337.09
3,4,11,revenue,1976.63,5214.47,18503.34
4,5,3,expense,123.65,9741.72,15748.88
...,...,...,...,...,...,...
295,296,3,expense,164.91,9741.72,15748.88
296,297,8,expense,245.34,14745.46,19160.74
297,298,5,revenue,1588.89,10509.73,20089.33
298,299,9,expense,220.67,6354.36,21400.09


---
## Aufgabe 3 - Datenklasse und Fehlerfall

### Aufgabe 3 - Kleine Datenklasse bauen und Fehler pruefen

Sie benoetigen nicht den gesamten Datensatz als Modell. In dieser Aufgabe erstellen Sie eine kleinere Datenklasse fuer den spaeteren Datenbank-Import.

**a)** Erstellen Sie selbst eine Datenklasse fuer eine einzelne Buchung.

**b)** Verwenden Sie nur einen kleinen, sinnvollen Ausschnitt des Datensatzes.

**c)** Erstellen Sie einen gueltigen Test-Payload und einen fehlerhaften Test-Payload.

**d)** Testen Sie, was passiert, wenn Sie den fehlerhaften Payload ohne `pydantic` in Ihre Datenklasse uebernehmen.

**e)** Erstellen Sie danach ein passendes `pydantic`-Modell mit denselben Kernfeldern.

**f)** Validieren Sie damit den gleichen Fehlerfall noch einmal.

**g)** Bereiten Sie die gueltigen Datensaetze fuer den spaeteren DB-Insert vor.

<details>
<summary><b>Tipp zu a) - Klein anfangen</b></summary>

Ein sinnvolles kleines Format waere zum Beispiel mit diesen Feldern: `booking_id`, `booking_date`, `booking_type`, `category`, `amount_net`, `is_paid`.
Beispielidee:
```python
from dataclasses import dataclass

@dataclass
class SmallBooking:
    ...
```
</details>

<details>
<summary><b>Tipp zu b) - Reale Daten als Vorlage</b></summary>

Sie koennen die ersten Zeilen des CSV nutzen und nur die benoetigten Spalten herausziehen.
</details>

<details>
<summary><b>Tipp zu c) und d) - Typfehler bewusst erzeugen</b></summary>

Geeignete Testfehler sind zum Beispiel: `amount_net` als Text, `is_paid` als String oder ein fehlendes Feld.
</details>

<details>
<summary><b>Tipp zu e) und f) - Gleicher Datenfall, anderer Ansatz</b></summary>

Verwenden Sie fuer den Vergleich denselben gueltigen und denselben fehlerhaften Payload.
Beispielidee:
```python
from pydantic import BaseModel

class SmallBookingModel(BaseModel):
    ...
```
oder
```python
from pydantic.dataclasses import dataclass

@dataclass
class SmallBookingModel:
    ...
```
</details>

In [ ]:
# Startpunkt fuer Aufgabe 3

small_columns = [
    "booking_id",
    "booking_date",
    "booking_type",
    "category",
    "amount_net",
    "is_paid",
]

sample_payload = df.loc[0, small_columns].to_dict()
bad_payload = sample_payload.copy()
bad_payload["amount_net"] = "812 EUR"
bad_payload["is_paid"] = "offen"

print(sample_payload)
print(bad_payload)

#a)
from dataclasses import dataclass
from datetime import date
from pydantic import BaseModel, ValidationError

@dataclass
class buchungen:
    booking_id: int
    booking_date: date
    booking_type: str
    category: str
    amount_net: float
    is_paid: bool

#b)
df_bookings = df[small_columns].copy()
df_bookings["amount_net"] = pd.to_numeric(df_bookings["amount_net"], errors="coerce")
df_bookings["is_paid"] = df_bookings["is_paid"].map({"bezahlt": True, "offen": False})
display(df_bookings.head(3))

#c)
gültig = {
    "booking_id": 100,
    "booking_date": "2025-01-01",
    "booking_type": "expense",
    "category": "service",
    "amount_net": 812.0,
    "is_paid": False
}
ungültig = {
    "booking_id": 100,
    "booking_date": "2025-01-01",
    "booking_type": "expense",
    "category": "service",
    "amount_net": "812 EUR",
    "is_paid": "offen"
}
#d)
class buchungen(BaseModel):
    booking_id: int
    booking_date: date
    booking_type: str
    category: str
    amount_net: float
    is_paid: bool
buchungen_gültig = buchungen(**gültig)
buchungen_ungültig = buchungen(**ungültig)  

SyntaxError: invalid syntax (155085325.py, line 23)

---
## Aufgabe 4 - SQLite mit kleinem Datenformat

### Aufgabe 4 - Tabelle passend zur Datenklasse anlegen und Daten einzeln laden

Die SQLite-Tabelle soll jetzt zum kleineren Datenformat aus Aufgabe 3 passen.

**a)** Erstellen Sie eine Tabelle, deren Spalten zu Ihrer Datenklasse passen.

**b)** Schreiben oder ergaenzen Sie eine Insert-Funktion fuer genau dieses Format.

**c)** Laden Sie mehrere Datensaetze einzeln in die Datenbank.

**d)** Lesen Sie die Tabelle wieder aus.

**e)** Testen Sie auch hier den Unterschied zwischen ungeprueften und validierten Daten.

<details>
<summary><b>Tipp zu a) - Datenbankschema klein halten</b></summary>

Die Tabelle soll nicht den gesamten CSV-Aufbau spiegeln, sondern nur Ihr kleineres Importformat.
Beispielidee:
```python
CREATE TABLE IF NOT EXISTS small_bookings (
    booking_id INTEGER PRIMARY KEY,
    booking_date TEXT NOT NULL,
    booking_type TEXT NOT NULL,
    category TEXT NOT NULL,
    amount_net REAL NOT NULL,
    is_paid INTEGER NOT NULL
)
```
</details>

<details>
<summary><b>Tipp zu b) - Spaltenreihenfolge vergleichen</b></summary>

Die Reihenfolge in `INSERT INTO ...` und die Reihenfolge der uebergebenen Werte muessen zusammenpassen.
Beispielidee:
```python
def insert_small_booking(connection, booking_row):
    connection.execute(
        "INSERT INTO small_bookings (...) VALUES (?, ?, ?, ?, ?, ?)",
        (...)
    )
```
</details>

<details>
<summary><b>Tipp zu c) - Einzelne Inserts bewusst nutzen</b></summary>

In dieser Aufgabe sollen die Datensaetze nacheinander geladen werden, damit Sie Fehlerfaelle besser beobachten koennen.
</details>

<details>
<summary><b>Tipp zu d) - Ruecklesen nicht vergessen</b></summary>

Mit `pd.read_sql_query(...)` koennen Sie sehr schnell pruefen, welche Datensaetze wirklich angekommen sind.
</details>

In [42]:
import sqlite3
from unicodedata import category

conn = sqlite3.connect("abv6_small_bookings.db")

rows_for_db = df.loc[:4, [
    "booking_id",
    "booking_date",
    "booking_type",
    "category",
    "amount_net",
    "is_paid",
]].copy()

small_bookings = {
    "booking_id": 3,
    "booking_date": "2025-01-01",
    "booking_type": "expense",
    "category": "service",
    "amount_net": 812.0,
    "is_paid": False
}

display(rows_for_db)

from pydantic import BaseModel, ValidationError
from datetime import date
class BookingInput(BaseModel):
    booking_id: int
    booking_date: date
    booking_type: str
    category: str
    amount_net: float
    is_paid: bool

booking = BookingInput(**small_bookings)
BookingInput.ValidationError(small_booking)

,booking_id,booking_date,booking_type,category,amount_net,is_paid
0,1,2025-05-21,expense,cogs,629.38,True
1,2,2025-12-26,revenue,other_income,1312.64,True
2,3,2025-07-14,revenue,service,1157.85,False
3,4,2025-11-13,revenue,other_income,1976.63,True
4,5,2025-03-25,expense,software,123.65,True


AttributeError: ValidationError

---
## Hausaufgabe - KPI aus der Datenbank und Vergleich mit neuem Datensatz

Nutzen Sie die Datenbank aus Aufgabe 4 oder laden Sie den gesamten Datensatz `buchungen_2025.csv` mit `to_sql()` in eine SQLite-Datenbank.

**Ziel:** Erstellen Sie eine Funktion, die eine Finanzkennzahl aus der Datenbank ausliest und berechnet.

**Vorschlag fuer eine KPI:** `operating_margin`

Formel:
`operating_margin = (revenue_total - expense_total) / revenue_total`

**Aufgaben:**
1. Laden Sie den Datensatz in eine SQLite-Tabelle, falls er noch nicht in der DB liegt.
2. Schreiben Sie eine Funktion, die `revenue_total`, `expense_total` und `operating_margin` aus der DB liest bzw. berechnet.
3. Geben Sie die Kennzahl fuer den aktuellen Datensatz aus.
4. Nutzen Sie den Datensatzgenerator aus der vorletzten Woche und erzeugen Sie 300 neue Datenpunkte.
5. Laden Sie den neuen Datensatz ebenfalls in die Datenbank.
6. Berechnen Sie die gleiche KPI erneut.
7. Vergleichen Sie beide Ergebnisse.

<details>
<summary><b>Tipp zu 1) - to_sql als schneller Einstieg</b></summary>

Wenn Sie den ganzen DataFrame in eine Tabelle schreiben wollen, ist `df.to_sql(...)` oft der schnellste Startpunkt.
Beispielidee:
```python
df.to_sql("bookings_full", ha_conn, if_exists="replace", index=False)
```
</details>

<details>
<summary><b>Tipp zu 2) - KPI in zwei Schritten denken</b></summary>

Lesen Sie zuerst Umsatz und Aufwand getrennt aus der Datenbank und berechnen Sie die Kennzahl danach in Python.
Beispielidee:
```python
def compute_operating_margin(connection, table_name):
    ...
```
</details>

<details>
<summary><b>Tipp zu 4) - Generator ausfuehren</b></summary>

Verwenden Sie den vorhandenen Generator `generate_dataset.py`.
Pruefen Sie danach, in welche CSV-Datei die neuen Daten geschrieben wurden.
</details>

<details>
<summary><b>Tipp zu 7) - Gleiche Funktion fuer beide Datensaetze</b></summary>

Wenn Sie fuer beide Tabellen dieselbe KPI-Funktion verwenden, wird der Vergleich einfacher.
</details>

In [ ]:
# Startpunkt fuer die Hausaufgabe

import sqlite3
from pathlib import Path

ha_conn = sqlite3.connect("abv6_homework.db")